# Model 1 Performance

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import *

DATA_PATH = Path('engineered_dataset.csv')

# If files are loaded in repository layout hide above and unmark below:
# DATA_PATH = Path('/.../.../Datasets/Engineered/engineered_dataset.csv')

df = pd.read_csv(DATA_PATH)

X=df.drop(columns=['returned','order_id']); y=df['returned'].astype(int)
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.20,random_state=42,stratify=y)
num=X.select_dtypes(include=['number']).columns.tolist(); cat=X.select_dtypes(exclude=['number']).columns.tolist()


In [ ]:
pre=ColumnTransformer([('num',Pipeline([('imp',SimpleImputer(strategy='median')),('sc',StandardScaler())]),num),
                       ('cat',Pipeline([('imp',SimpleImputer(strategy='most_frequent')),('oh',OneHotEncoder(handle_unknown='ignore'))]),cat)])
model=Pipeline([('preprocessor',pre),('classifier',LogisticRegression(C=1.0,penalty='l2',solver='liblinear',class_weight='balanced',max_iter=1000,random_state=42))])
model.fit(X_train,y_train)

In [ ]:
pred=model.predict(X_test); prob=model.predict_proba(X_test)[:,1]
results=pd.DataFrame([{'Accuracy':accuracy_score(y_test,pred),'Precision':precision_score(y_test,pred,zero_division=0),'Recall':recall_score(y_test,pred,zero_division=0),'F1-score':f1_score(y_test,pred,zero_division=0),'ROC-AUC':roc_auc_score(y_test,prob)}])
print(results.to_string(index=False))
print('\nConfusion matrix:\n',confusion_matrix(y_test,pred))